In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

In [9]:
X,y = make_classification(n_samples=1000,
                          n_features=10,
                          n_redundant=8,
                          weights=[0.9,0.1],
                          flip_y=0,
                          random_state=42)
np.unique(y,return_counts=True)

(array([0, 1]), array([900, 100]))

In [10]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,stratify=y,random_state=42)

#### Experiment 1: Train Logistic Regression

In [11]:
params={
    'solver':'lbfgs',
    'max_iter':1000,
    'multi_class':'auto',
    'random_state':8888
}
lr=LogisticRegression(**params)
lr.fit(X_train,y_train)
y_pred=lr.predict(X_test)
report=classification_report(y_test,y_pred)
print(report)

              precision    recall  f1-score   support

           0       0.95      0.97      0.96       270
           1       0.62      0.50      0.56        30

    accuracy                           0.92       300
   macro avg       0.79      0.73      0.76       300
weighted avg       0.91      0.92      0.92       300



In [12]:
report_dict=classification_report(y_test,y_pred,output_dict=True)
report_dict

{'0': {'precision': 0.9456521739130435,
  'recall': 0.9666666666666667,
  'f1-score': 0.9560439560439561,
  'support': 270.0},
 '1': {'precision': 0.625,
  'recall': 0.5,
  'f1-score': 0.5555555555555556,
  'support': 30.0},
 'accuracy': 0.92,
 'macro avg': {'precision': 0.7853260869565217,
  'recall': 0.7333333333333334,
  'f1-score': 0.7557997557997558,
  'support': 300.0},
 'weighted avg': {'precision': 0.9135869565217392,
  'recall': 0.92,
  'f1-score': 0.9159951159951161,
  'support': 300.0}}

In [13]:
import mlflow

In [14]:
mlflow.set_experiment('First Experiment')
mlflow.set_tracking_uri('http://127.0.0.1:5000/')

with mlflow.start_run():
    mlflow.log_params(params)
    mlflow.log_metrics({
        'accuracy':report_dict['accuracy'],
        'recall_class_0':report_dict['0']['recall'],
        'recall_class_1':report_dict['1']['recall'],
        'f1_score_class_0':report_dict['0']['f1-score'],
        'f1_score_class_1':report_dict['1']['f1-score'],
        'f1_score_macro_avg':report_dict['macro avg']['f1-score']
    })
    mlflow.sklearn.log_model(lr,'Logistic Regression')

2025/11/25 20:47:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/25 20:47:36 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run abundant-turtle-659 at: http://127.0.0.1:5000/#/experiments/192307785387825820/runs/57f2e5b096e5439abd1056be653ded14
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/192307785387825820


#### Experiment 2: Train Random Forest Classifier

In [16]:
rf_clf=RandomForestClassifier(n_estimators=30,max_depth=3)
rf_clf.fit(X_train,y_train)
y_pred_rf=rf_clf.predict(X_test)
print(classification_report(y_test,y_pred_rf))

              precision    recall  f1-score   support

           0       0.96      1.00      0.98       270
           1       0.95      0.67      0.78        30

    accuracy                           0.96       300
   macro avg       0.96      0.83      0.88       300
weighted avg       0.96      0.96      0.96       300



#### Experiment 3: Train XGBoost

In [17]:
xgb_clf=XGBClassifier(use_label_encoder=False,eval_metrics='logloss')
xgb_clf.fit(X_train,y_train)
y_pred_xgb=xgb_clf.predict(X_test)
print(classification_report(y_test,y_pred_xgb))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99       270
           1       0.96      0.80      0.87        30

    accuracy                           0.98       300
   macro avg       0.97      0.90      0.93       300
weighted avg       0.98      0.98      0.98       300



#### Experiment 4: Handle class Imbalance using SMOTETomek and then train XGBoost

In [20]:
pip install imblearn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
from imblearn.combine import SMOTETomek

sat= SMOTETomek(random_state=42)
X_train_res,y_train_res = sat.fit_resample(X_train,y_train)
np.unique(y_train_res,return_counts=True)

(array([0, 1]), array([619, 619]))

In [24]:
xgb_clf.fit(X_train_res,y_train_res)
y_pred_xgb_smote=xgb_clf.predict(X_test)
print(classification_report(y_test,y_pred_xgb_smote))

              precision    recall  f1-score   support

           0       0.98      0.98      0.98       270
           1       0.81      0.83      0.82        30

    accuracy                           0.96       300
   macro avg       0.89      0.91      0.90       300
weighted avg       0.96      0.96      0.96       300



In [25]:
models=[
    (
      'LogisticRegression',
        LogisticRegression(C=1,solver='liblinear'),
        (X_train,y_train),
        (X_test,y_test)
    ),
    (
        'Random Forest',
        RandomForestClassifier(n_estimators=30,max_depth=3),
        (X_train,y_train),
        (X_test,y_test)
    ),
    (
        'XGBClassifier',
        XGBClassifier(use_label_encoder=False,eval_metrics='logloss'),
        (X_train,y_train),
        (X_test,y_test)
    ),
    (
        'XGBClassifier with SMOTE',
        XGBClassifier(use_label_encoder=False,eval_metrics='logloss'),
        (X_train_res,y_train_res),
        (X_test,y_test)
    )
]

In [27]:
reports= []
for model_name,model,train_set,test_set in models:
    X_train = train_set[0]
    y_train = train_set[1]
    X_test = test_set[0]
    y_test = test_set[1]

    model.fit(X_train,y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test,y_pred,output_dict=True)
    reports.append(report)

In [28]:
reports

[{'0': {'precision': 0.9454545454545454,
   'recall': 0.9629629629629629,
   'f1-score': 0.9541284403669725,
   'support': 270.0},
  '1': {'precision': 0.6,
   'recall': 0.5,
   'f1-score': 0.5454545454545454,
   'support': 30.0},
  'accuracy': 0.9166666666666666,
  'macro avg': {'precision': 0.7727272727272727,
   'recall': 0.7314814814814814,
   'f1-score': 0.749791492910759,
   'support': 300.0},
  'weighted avg': {'precision': 0.9109090909090909,
   'recall': 0.9166666666666666,
   'f1-score': 0.91326105087573,
   'support': 300.0}},
 {'0': {'precision': 0.9676258992805755,
   'recall': 0.9962962962962963,
   'f1-score': 0.9817518248175182,
   'support': 270.0},
  '1': {'precision': 0.9545454545454546,
   'recall': 0.7,
   'f1-score': 0.8076923076923077,
   'support': 30.0},
  'accuracy': 0.9666666666666667,
  'macro avg': {'precision': 0.961085676913015,
   'recall': 0.8481481481481481,
   'f1-score': 0.8947220662549129,
   'support': 300.0},
  'weighted avg': {'precision': 0.9663

In [32]:
mlflow.set_experiment('Anomaly_detection')
mlflow.set_tracking_uri('http://127.0.0.1:5000/')

for i,element in enumerate(models):
    model_name=element[0]
    model=element[1]
    report=reports[i]
    with mlflow.start_run(run_name=model_name):
        mlflow.log_param('model_name',model_name)
        
        mlflow.log_metrics({
        'accuracy':report['accuracy'],
        'recall_class_0':report['0']['recall'],
        'recall_class_1':report['1']['recall'],
        'f1_score_class_0':report['0']['f1-score'],
        'f1_score_class_1':report['1']['f1-score'],
        'f1_score_macro_avg':report['macro avg']['f1-score']
         })
        if 'XGB' in model_name:
             mlflow.xgboost.log_model(model,'model')
        else:
             mlflow.sklearn.log_model(model,'model')


2025/11/25 22:56:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/25 22:56:30 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/11/25 22:56:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run LogisticRegression at: http://127.0.0.1:5000/#/experiments/801989169201354244/runs/194138c377624ea8801c7f9ac6627364
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/801989169201354244


2025/11/25 22:56:39 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run Random Forest at: http://127.0.0.1:5000/#/experiments/801989169201354244/runs/47661d5693434aa2997d98e7572a4d3b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/801989169201354244


2025/11/25 22:56:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/25 22:56:51 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/11/25 22:56:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at: http://127.0.0.1:5000/#/experiments/801989169201354244/runs/b858dbf18f8e4ebebe4b94956621ab39
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/801989169201354244


2025/11/25 22:57:02 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run XGBClassifier with SMOTE at: http://127.0.0.1:5000/#/experiments/801989169201354244/runs/93b46ddc39a34251bbf0584395144276
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/801989169201354244
